# CI/CD & Packaging


## What you get

| Path | What it is |
|---|---|
| `src/forecasting/data_helpers.py` | First piece of a shared package -- `fill_series_gaps`, extracted from notebook 5's Step 1 and parameterized instead of relying on notebook globals. |
| `tests/test_data_helpers.py` | Two real unit tests against it. |
| `tests/test_pipeline_compiles.py` | Extracts notebook 6's actual pipeline code and compiles it with the real `kfp` SDK -- automates the exact manual check used throughout this project's development. |
| `ci/compile_pipeline.py` | The same extraction+compile, but writing the result out -- what the deploy workflow runs. |
| `ci/push_pipeline_template.py` | Pushes that compiled template to Artifact Registry. |
| `.github/workflows/ci.yml` | Lint + test, on every PR and push to main. |
| `.github/workflows/deploy-pipeline.yml` | Recompile + push, on merge to main -- needs GCP setup first, see the cell below. |
| `requirements-dev.txt` | Lint/test tooling versions, pinned to what was actually used to verify all of this. |
| `requirements.txt` | The notebooks' own runtime dependencies (0-8) -- separate from `requirements-dev.txt`, which is CI-only. |
| `README.md` / `docs/ARCHITECTURE.md` | Full-project overview and design-decision writeup covering notebooks 0-8, the `functions/` package, and `config/params.yaml` -- meant to be read before an interview walkthrough, not just left in the repo. |

**What this doesn't do:** touch any of the actual research/pipeline notebooks. It only adds new files alongside them.

In [ ]:
import pathlib

## src/forecasting/: First piece of the shared package

The roadmap's project-scaffold phase calls for a `src/` package instead
of each notebook redefining its own copies of the same logic. 


In [ ]:
pathlib.Path("src/forecasting/__init__.py").parent.mkdir(parents=True, exist_ok=True)
pathlib.Path("src/forecasting/__init__.py").write_text('')
print("wrote src/forecasting/__init__.py")

In [ ]:
pathlib.Path("src/forecasting/data_helpers.py").parent.mkdir(parents=True, exist_ok=True)
pathlib.Path("src/forecasting/data_helpers.py").write_text('"""Shared data-prep helpers for the forecasting project.\n\nExtracted from 5_Model_Comparison_ABC_Category.ipynb\'s Step 1 (local\nAutoARIMA) section, where it was originally a notebook-global function\nrelying on TIME_COLUMN/TARGET_COLUMN/SERIES_ID_COLUMN being set elsewhere\nin that notebook. Refactored here to take those as explicit parameters\ninstead, since a standalone module can\'t rely on notebook globals.\n\nThis is the first piece of the "src/ shared package" the roadmap names\n(the project-scaffold phase) -- nothing else has been extracted yet, and\nthe notebooks still define their own copies rather than importing from\nhere. Wiring the notebooks to import this instead of redefining it is a\nseparate follow-up, not done as part of this pass.\n"""\n\nimport pandas as pd\n\n\ndef fill_series_gaps(\n    sku_df: pd.DataFrame,\n    sku: str,\n    time_column: str = "date",\n    target_column: str = "sales_qty",\n    series_id_column: str = "sku_id",\n) -> pd.DataFrame:\n    """Reindex one SKU\'s rows onto a gap-free date grid, filling missing weeks with 0.\n\n    Takes `sku` explicitly rather than reading it back off `sku_df` after\n    reindexing -- reindexing onto the gap-filled date grid introduces new\n    rows for the missing weeks, and every column except the one explicitly\n    .fillna(0) below (target_column) comes back NaN for those new rows,\n    including the id column itself. Caught via a synthetic-data test with\n    a deliberate gap before this was first shipped -- it silently produced\n    a NaN id for every gap-filled row, which downstream per-series tooling\n    (statsforecast, darts) chokes on with a string-vs-float comparison\n    error.\n    """\n    sku_df = sku_df.sort_values(time_column).set_index(time_column)\n    freq = pd.infer_freq(sku_df.index)\n    if freq is None:\n        anchor = pd.Series(sku_df.index.day_name()).mode().iat[0][:3].upper()\n        freq = f"W-{anchor}"\n    full_index = pd.date_range(sku_df.index.min(), sku_df.index.max(), freq=freq)\n    sku_df = sku_df.reindex(full_index)\n    sku_df[target_column] = sku_df[target_column].fillna(0)\n    sku_df[series_id_column] = sku  # restore the id for the newly-introduced gap rows\n    sku_df.index.name = time_column\n    return sku_df.reset_index()\n')
print("wrote src/forecasting/data_helpers.py")

## tests/: Real tests, run locally before this notebook shipped

`test_data_helpers.py` covers the exact bug `fill_series_gaps` exists to
prevent (a gap-filled row's id column coming back `NaN`) plus a no-op
sanity check on already-regular data.

In [ ]:
pathlib.Path("tests/test_data_helpers.py").parent.mkdir(parents=True, exist_ok=True)
pathlib.Path("tests/test_data_helpers.py").write_text('import pandas as pd\n\nfrom forecasting.data_helpers import fill_series_gaps\n\n\ndef test_fill_series_gaps_fills_missing_weeks_with_zero_and_keeps_id():\n    all_dates = pd.date_range("2023-01-02", periods=6, freq="W-MON")\n    gap_dates = all_dates.delete([2, 3])  # drop two weeks in the middle -> a real gap\n    sku_df = pd.DataFrame({\n        "date": gap_dates,\n        "sku_id": "SKU_TEST",\n        "sales_qty": [10, 20, 30, 40],\n    })\n\n    filled = fill_series_gaps(sku_df, "SKU_TEST")\n\n    assert len(filled) == 6  # both gap weeks restored\n    assert filled["sku_id"].isna().sum() == 0  # the exact bug this function exists to prevent\n    assert (filled["sku_id"] == "SKU_TEST").all()\n\n    gap_rows = filled[filled["date"].isin(all_dates[[2, 3]])]\n    assert len(gap_rows) == 2\n    assert (gap_rows["sales_qty"] == 0).all()\n\n\ndef test_fill_series_gaps_is_a_noop_on_already_gap_free_data():\n    dates = pd.date_range("2023-01-02", periods=4, freq="W-MON")\n    sku_df = pd.DataFrame({"date": dates, "sku_id": "SKU_TEST", "sales_qty": [1, 2, 3, 4]})\n\n    filled = fill_series_gaps(sku_df, "SKU_TEST")\n\n    assert len(filled) == 4\n    assert list(filled["sales_qty"]) == [1, 2, 3, 4]\n')
print("wrote tests/test_data_helpers.py")

In [ ]:
pathlib.Path("tests/test_pipeline_compiles.py").parent.mkdir(parents=True, exist_ok=True)
pathlib.Path("tests/test_pipeline_compiles.py").write_text('"""Confidence check for 6_Kubeflow_Pipelines.ipynb: does the pipeline still compile?\n\nAutomates the same manual check performed by hand throughout this\nproject\'s development: extract the notebook\'s component + pipeline\ndefinitions, feed them to the real kfp SDK, and confirm\ncompiler.Compiler().compile(...) succeeds and produces the DAG edges this\npipeline depends on for correctness. Catches exactly the class of bug that\ncaused the original `AttributeError: \'PipelineArtifactChannel\' object has\nno attribute \'uri\'` regression -- something that only fails at pipeline\n*definition* time, which notebook syntax checks alone won\'t catch.\n"""\n\nimport pathlib\nimport tempfile\n\nimport nbformat\nimport pytest\nimport yaml\n\nNOTEBOOK_PATH = pathlib.Path(__file__).parents[1] / "6_Kubeflow_Pipelines.ipynb"\n\n\ndef _load_relevant_code_cells():\n    nb = nbformat.read(NOTEBOOK_PATH, as_version=4)\n    keep = []\n    for cell in nb.cells:\n        if cell.cell_type != "code":\n            continue\n        src = cell.source\n        if src.lstrip().startswith("project = !gcloud"):\n            continue  # shell-magic project lookup -- not needed to compile\n        if src.lstrip().startswith("compiler.Compiler().compile"):\n            break  # stop before the notebook\'s own compile+submit cell; this test does its own compile\n        keep.append(src)\n    return keep\n\n\n@pytest.mark.skipif(not NOTEBOOK_PATH.exists(), reason="6_Kubeflow_Pipelines.ipynb not found")\ndef test_pipeline_compiles_and_has_expected_dag_edges():\n    namespace: dict = {}\n    for src in _load_relevant_code_cells():\n        # exec is the point, not a shortcut around one: this is how the\n        # notebook\'s own component/pipeline definitions become real Python\n        # objects to compile, without hand-duplicating that code here. The\n        # source is this repo\'s own tracked notebook, not untrusted input.\n        exec(compile(src, str(NOTEBOOK_PATH), "exec"), namespace)  # noqa: S102\n\n    compiler = namespace["compiler"]\n    forecasting_pipeline = namespace["forecasting_pipeline"]\n\n    with tempfile.NamedTemporaryFile(suffix=".yaml") as f:\n        compiler.Compiler().compile(pipeline_func=forecasting_pipeline, package_path=f.name)\n        with open(f.name) as fh:\n            spec = yaml.safe_load(fh)\n\n    exit_handler_dag = next(\n        c["dag"] for name, c in spec["components"].items() if "exit-handler" in name\n    )\n    tasks = exit_handler_dag["tasks"]\n\n    # The dependency this whole pipeline exists to get right: eval must not\n    # run before training actually produces a real WAPE to compare against.\n    assert tasks["evaluate-and-compare-op"]["dependentTasks"] == ["train-xgboost-candidate-op"]\n    assert tasks["train-xgboost-candidate-op"]["dependentTasks"] == ["prepare-data-op"]\n')
print("wrote tests/test_pipeline_compiles.py")

### Prove it locally

Run this from your repo root once the cells above have written the files

In [ ]:
import subprocess

print(subprocess.run(
    ["python", "-m", "pytest", "tests/", "-v"],
    env={"PYTHONPATH": "src", **__import__("os").environ},
    capture_output=True, text=True,
).stdout)

## ci/: GitHub Actions workflows call

In [ ]:
pathlib.Path("ci/compile_pipeline.py").parent.mkdir(parents=True, exist_ok=True)
pathlib.Path("ci/compile_pipeline.py").write_text('"""Recompile 6_Kubeflow_Pipelines.ipynb\'s pipeline definition to a template file.\n\nUsed by the deploy-pipeline GitHub Actions workflow on every merge to\nmain -- this is the actual "redeploy the pipeline definition" step from\nthe CI/CD roadmap phase. Shares its notebook-cell-extraction logic with\ntests/test_pipeline_compiles.py (that test IS this same compile, just\nasserting on the result instead of writing it out) -- known duplication,\nflagged rather than hidden: if you change how one extracts cells, change\nthe other the same way, or fold both into one shared helper later.\n"""\n\nimport pathlib\n\nimport nbformat\n\nNOTEBOOK_PATH = pathlib.Path(__file__).parents[1] / "6_Kubeflow_Pipelines.ipynb"\nOUTPUT_PATH = pathlib.Path(__file__).parents[1] / "forecasting_pipeline.yaml"\n\n\ndef _load_relevant_code_cells():\n    nb = nbformat.read(NOTEBOOK_PATH, as_version=4)\n    keep = []\n    for cell in nb.cells:\n        if cell.cell_type != "code":\n            continue\n        src = cell.source\n        if src.lstrip().startswith("project = !gcloud"):\n            continue  # shell-magic project lookup -- not needed to compile\n        if src.lstrip().startswith("compiler.Compiler().compile"):\n            break  # stop before the notebook\'s own compile+submit cell; this script does its own\n        keep.append(src)\n    return keep\n\n\ndef main():\n    namespace: dict = {}\n    for src in _load_relevant_code_cells():\n        # exec is the point, not a shortcut around one: this is how the\n        # notebook\'s own component/pipeline definitions become real Python\n        # objects to compile, without hand-duplicating that code here. The\n        # source is this repo\'s own tracked notebook, not untrusted input.\n        exec(compile(src, str(NOTEBOOK_PATH), "exec"), namespace)  # noqa: S102\n\n    compiler = namespace["compiler"]\n    forecasting_pipeline = namespace["forecasting_pipeline"]\n    compiler.Compiler().compile(pipeline_func=forecasting_pipeline, package_path=str(OUTPUT_PATH))\n    print(f"Compiled {NOTEBOOK_PATH.name} -> {OUTPUT_PATH.name}")\n\n\nif __name__ == "__main__":\n    main()\n')
print("wrote ci/compile_pipeline.py")

In [ ]:
pathlib.Path("ci/push_pipeline_template.py").parent.mkdir(parents=True, exist_ok=True)
pathlib.Path("ci/push_pipeline_template.py").write_text('"""Push the compiled pipeline template to an Artifact Registry KFP repo.\n\nRun after ci/compile_pipeline.py, in the deploy-pipeline GitHub Actions\nworkflow. Reads PROJECT_ID/REGION/REPO from the environment (set as env:\nin that workflow) rather than hardcoding them, since this script has no\nother way to know which GCP project it\'s running against -- authentication\n(google-github-actions/auth, Workload Identity Federation) happens as a\nseparate step before this runs; RegistryClient picks up those Application\nDefault Credentials automatically.\n"""\n\nimport os\nimport pathlib\n\nfrom kfp.registry import RegistryClient\n\nPIPELINE_PATH = pathlib.Path(__file__).parents[1] / "forecasting_pipeline.yaml"\n\n\ndef main():\n    project_id = os.environ["PROJECT_ID"]\n    region = os.environ["REGION"]\n    repo = os.environ["REPO"]\n    commit_sha = os.environ.get("GITHUB_SHA", "manual")[:12]\n\n    client = RegistryClient(host=f"https://{region}-kfp.pkg.dev/{project_id}/{repo}")\n    template_name, version_name = client.upload_pipeline(\n        file_name=str(PIPELINE_PATH),\n        tags=["latest", commit_sha],\n    )\n    print(f"Pushed {template_name} ({version_name}) tagged latest, {commit_sha}")\n\n\nif __name__ == "__main__":\n    main()\n')
print("wrote ci/push_pipeline_template.py")

### Prove the compile step locally

In [ ]:
import subprocess

result = subprocess.run(["python", "ci/compile_pipeline.py"], capture_output=True, text=True)
print(result.stdout or result.stderr)

## .github/workflows/: 2 pieces of CI/CD

In [ ]:
pathlib.Path(".github/workflows/ci.yml").parent.mkdir(parents=True, exist_ok=True)
pathlib.Path(".github/workflows/ci.yml").write_text('name: CI\n\n# Runs on every PR and every push to main: lint, then run whatever unit\n# tests exist. Right now that\'s fill_series_gaps\' two tests and the\n# pipeline-compiles check -- more get added as more of src/ exists.\non:\n  pull_request:\n    branches: [main]\n  push:\n    branches: [main]\n\njobs:\n  lint-and-test:\n    runs-on: ubuntu-latest\n    steps:\n      - uses: actions/checkout@v7\n\n      - uses: actions/setup-python@v6\n        with:\n          python-version: "3.11"\n\n      - name: Install dependencies\n        run: pip install -r requirements-dev.txt\n\n      - name: Lint\n        run: ruff check src/ tests/ ci/\n\n      - name: Run tests\n        env:\n          PYTHONPATH: src\n        run: pytest tests/ -v\n')
print("wrote .github/workflows/ci.yml")

In [ ]:
pathlib.Path(".github/workflows/deploy-pipeline.yml").parent.mkdir(parents=True, exist_ok=True)
pathlib.Path(".github/workflows/deploy-pipeline.yml").write_text('name: Deploy Pipeline\n\n# Redeploys the pipeline *definition* on merge to main -- recompiles\n# 6_Kubeflow_Pipelines.ipynb and pushes the result to an Artifact Registry\n# KFP template repo. This does NOT run the pipeline (that\'s Step 7\'s\n# PipelineJobSchedule, a separate Vertex-side resource) -- it only updates\n# which template that schedule (or a manual run) picks up next time.\n#\n# SETUP THIS NEEDS BEFORE IT WILL WORK (none of this exists yet):\n#   1. An Artifact Registry repo of type "KFP template" -- e.g.\n#      `gcloud artifacts repositories create forecasting-pipelines \\\n#         --repository-format=kfp --location=us-central1`\n#   2. Workload Identity Federation, not a long-lived service account key --\n#      a GitHub-to-GCP WIF pool/provider, and a service account it can\n#      impersonate with permission to push to that repo. See\n#      google-github-actions/auth\'s own README for the one-time GCP-side\n#      setup: https://github.com/google-github-actions/auth\n#   3. Three values from that setup, stored as repo secrets:\n#      GCP_WORKLOAD_IDENTITY_PROVIDER, GCP_SERVICE_ACCOUNT, GCP_PROJECT_ID\non:\n  push:\n    branches: [main]\n    paths:\n      - "6_Kubeflow_Pipelines.ipynb"\n      - "ci/compile_pipeline.py"\n\njobs:\n  compile-and-push:\n    runs-on: ubuntu-latest\n    permissions:\n      contents: read\n      id-token: write # required for Workload Identity Federation\n    steps:\n      - uses: actions/checkout@v7\n\n      - uses: actions/setup-python@v6\n        with:\n          python-version: "3.11"\n\n      - name: Install dependencies\n        run: pip install -r requirements-dev.txt\n\n      - id: auth\n        uses: google-github-actions/auth@v3\n        with:\n          workload_identity_provider: ${{ secrets.GCP_WORKLOAD_IDENTITY_PROVIDER }}\n          service_account: ${{ secrets.GCP_SERVICE_ACCOUNT }}\n\n      - name: Compile pipeline definition\n        run: python ci/compile_pipeline.py\n\n      - name: Push to Artifact Registry\n        env:\n          PROJECT_ID: ${{ secrets.GCP_PROJECT_ID }}\n          REGION: us-central1\n          REPO: forecasting-pipelines\n        run: python ci/push_pipeline_template.py\n')
print("wrote .github/workflows/deploy-pipeline.yml")

## Everything else

Including .gitignore, git add,  __pycache__/,.pytest_cache/, .ruff_cache/, and the compiled forecasting_pipeline.yaml build artifact

In [ ]:
pathlib.Path(".gitignore").parent.mkdir(parents=True, exist_ok=True)
pathlib.Path(".gitignore").write_text('# Build artifacts -- regenerated by ci/compile_pipeline.py, not source\nforecasting_pipeline.yaml\n\n# Python\n__pycache__/\n*.pyc\n*.egg-info/\n.eggs/\n\n# Test/lint caches\n.pytest_cache/\n.ruff_cache/\n\n# Jupyter\n.ipynb_checkpoints/\n\n# OS/editor cruft\n.DS_Store\n.vscode/\n\n# Never commit credentials -- this project authenticates via\n# Application Default Credentials / Workload Identity Federation, so none\n# of these should ever exist in the repo, but ignore them defensively.\n*.json.key\nservice-account*.json\n.env\n')
print("wrote .gitignore")

In [ ]:
pathlib.Path("requirements-dev.txt").parent.mkdir(parents=True, exist_ok=True)
pathlib.Path("requirements-dev.txt").write_text("# Lint/test tooling for CI -- not the notebooks' own runtime dependencies\n# (those are installed per-cell/per-component, closer to where they're used,\n# matching how this project has done it throughout).\npandas>=2.0\npytest>=8.0\nruff>=0.6\nnbformat>=5.9\nkfp>=2.9\npyyaml>=6.0\n")
print("wrote requirements-dev.txt")

### requirements.txt

In [ ]:
pathlib.Path('requirements.txt').parent.mkdir(parents=True, exist_ok=True)
pathlib.Path('requirements.txt').write_text("# Runtime dependencies for running the notebooks themselves (0-8) from a\n# local environment or a Vertex AI Workbench kernel -- NOT the same thing as\n# requirements-dev.txt, which is CI-only lint/test tooling for the\n# src/tests/ci/ scaffold that 8_CICD_and_Packaging.ipynb writes.\n#\n# What this deliberately does NOT include: anything a KFP component in\n# 6_Kubeflow_Pipelines.ipynb installs for itself via its own\n# `packages_to_install=[...]` (xgboost==1.6.2, setuptools<81, and each\n# component's own copy of the google-cloud-* clients). Each `@component` is\n# its own isolated container built from that list, not the notebook kernel's\n# environment -- installing xgboost here would do nothing for those and\n# would be misleading about where it's actually used.\n#\n# Versions are lower-bounded, not pinned, matching how this project has\n# installed everything throughout (`pip install <pkg> -U`) -- the one\n# exception is xgboost==1.6.2 inside 6_Kubeflow_Pipelines.ipynb's own\n# component definition, left untouched since that pin is load-bearing there\n# (see that notebook's markdown for why).\n\n# Core data handling -- every notebook\npandas>=2.0\nnumpy>=1.26\n\n# config/params.yaml -- the single source of truth for forecast_horizon /\n# validation_horizon / granularity, read by functions/params_utils.py\npyyaml>=6.0\n\n# GCP clients -- Vertex AI (datasets, training, endpoints, experiments,\n# model monitoring, pipeline schedules), Cloud Storage, BigQuery (+ the\n# Data Transfer Service client for the scheduled accuracy query),\n# Pub/Sub (pipeline-completion notifications)\ngoogle-cloud-aiplatform>=1.60\ngoogle-cloud-storage>=2.10\ngoogle-cloud-bigquery>=3.20\ngoogle-cloud-bigquery-datatransfer>=3.14\ngoogle-cloud-pubsub>=2.21\n\n# Vertex AI Pipelines (notebook 6): compiling/submitting the KFP DAG and the\n# VertexNotificationEmailOp exit handler -- not the per-component runtime\n# deps, which each component installs for itself (see note above)\nkfp>=2.9\ngoogle-cloud-pipeline-components>=2.14\ngoogleapis-common-protos>=1.63\n\n# Notebook 3 (EDA): STL decomposition, ADF/ACF/PACF stationarity checks\nstatsmodels>=0.14\n\n# Notebook 5 (Model Comparison): local open-source alternatives to\n# BQML ARIMA_PLUS / Vertex AI AutoML Forecasting, run by default to save\n# GCP credits (RUN_AUTOML defaults to False; RUN_LOCAL_ARIMA to True)\nstatsforecast>=1.7\ndarts[torch]>=0.41.0\ntorch>=2.0\n\n# Plotting -- notebooks 3, 4, 5\nmatplotlib>=3.8\nseaborn>=0.13\n")
print("wrote requirements.txt")

In [ ]:
pathlib.Path('README.md').parent.mkdir(parents=True, exist_ok=True)
pathlib.Path('README.md').write_text('# Fashion Demand Forecasting -- MLOps Pipeline\n\nA weekly SKU-level demand forecasting project on GCP, built specifically to\ngo deep on the parts of MLOps that are easy to skip -- feature-engineering\ncorrectness, experiment tracking, a real champion/challenger retraining\npipeline, monitoring, and CI/CD -- not just a notebook that produces a\nforecast once. It\'s a portfolio project, not a production system with a\nreal business behind it yet; see "What\'s real vs. placeholder" below before\nassuming everything here runs against live data.\n\nSynthetic data (300 SKUs, weekly, ~2.5 years) stands in for a real fashion\nretailer\'s sales history: prices, discounts, promotions, marketing spend,\nweather, product lifecycle, and a handful of endogenous demand signals\n(reviews, wishlist adds, social trend score, inventory/stockouts), all\ngenerated with a real underlying data-generating process (trend + seasonality\n+ price/promo/weather/marketing effects + lifecycle ramp-and-decline +\nnoise) in `1_Generate_Synthetic_Data.ipynb` -- not `np.random` with no\nstructure to actually learn from.\n\n## Notebooks\n\n| Notebook | What it does |\n|---|---|\n| `0_Setup.ipynb` | One-time GCP project setup: enable APIs, create the GCS bucket, IAM, package installs. |\n| `1_Generate_Synthetic_Data.ipynb` | Generates the synthetic dataset described above and uploads `config/params.yaml` -- the single source of truth for `forecast_horizon` (8 weeks), `validation_horizon` (104 weeks), and `granularity` ("W"), read everywhere else via `functions/params_utils.load_params()`. |\n| `2_Features.ipynb` | Runs the shared feature-engineering functions (`functions/Features_Functions.py`) over the raw data: calendar features, lifecycle stage, price features, target/endogenous lags and rolling stats, ABC/XYZ classification, one-hot encoding, and the time-based train/validation/test split. Also builds `future_df.csv` -- the `forecast_horizon` future rows per SKU that Vertex AI batch prediction scores against. |\n| `3_EDA.ipynb` | Exploratory analysis: target distribution, STL seasonal decomposition, stationarity (ADF/ACF/PACF), price/promo/weather relationships. |\n| `4_Baseline_Model.ipynb` | Trains the first model: Vertex AI **AutoML Forecasting** on the full feature set, evaluates it (RMSE/MAE/MAPE/WAPE), backtests against the TEST split, and pulls Shapley-value feature attributions. |\n| `5_Model_Comparison_ABC_Category.ipynb` | Sweeps three forecasting approaches -- AutoML Forecasting, a local `statsforecast` AutoARIMA (`RUN_LOCAL_ARIMA`, default on -- free, replaces BQML ARIMA_PLUS to save credits), and darts N-BEATS/N-HiTS -- across context windows, scored by WAPE at the overall/category/ABC-class level. AutoML itself stays gated behind `RUN_AUTOML = False` since it costs real money per run. |\n| `5.1_Experiment_Tracking.ipynb` | Logs every run from the comparison above to Vertex AI Experiments (the MLflow equivalent on GCP) so the sweep results are queryable later, not just printed once and lost. |\n| `6_Kubeflow_Pipelines.ipynb` | The retraining pipeline that actually runs in production: refresh data -> train an **XGBoost** candidate -> compare it against the current champion\'s real stored WAPE -> conditionally deploy to a Vertex AI Endpoint -> write an 8-week batch forecast to BigQuery -> notify by email/Pub-Sub. Also where the pipeline\'s weekly schedule lives. See "Two model tracks" below for why this trains its own XGBoost model rather than reusing AutoML. |\n| `7_Monitoring_and_Consumption.ipynb` | Reads what notebook 6 writes: querying the forecast tables (what a BI tool connects to), skew/drift monitoring on the deployed endpoint, and a scheduled realized-vs-predicted accuracy query. |\n| `8_CICD_and_Packaging.ipynb` | Writes out `src/`, `tests/`, `ci/`, `.github/workflows/`, and this README/requirements/architecture doc -- run once (or after a real change to the pipeline) to (re)generate the repo scaffold; not part of the pipeline itself. |\n\n## The `functions/` package\n\nShared, single-sourced logic imported by more than one notebook, instead of\neach notebook keeping its own copy:\n\n- **`Features_Functions.py`** -- every feature-engineering function (`add_calendar_features`, `add_lifecycle_features`, `add_price_features`, `add_target_lag_features`, `add_endogenous_lagged_features`, `add_cross_product_features`, `add_abc_classification`, `add_xyz_classification`, `encode_categoricals`, `time_based_split`), plus the covariate typing Vertex AI Forecasting needs: `AVAILABLE_AT_FORECAST` / `UNAVAILABLE_AT_FORECAST` (which columns must vs. must not carry a real value on a future batch-prediction row) and `build_column_specs()` (the categorical/numeric/timestamp map every AutoML Forecasting training job needs). `2_Features.ipynb`, `4_Baseline_Model.ipynb`, and `5_Model_Comparison_ABC_Category.ipynb` all import these rather than each hand-typing the same ~80-column lists -- see "Recently fixed" below for why that matters.\n- **`Metrics_Functions.py`** -- `wape`, `bias_pct`, `tracking_signal` (+ per-SKU/summary variants), `mase`, and a combined `evaluate()`.\n- **`params_utils.py`** -- `load_params()`, the one function that reads `gs://<bucket>/config/params.yaml`. Everything that needs `forecast_horizon`/`validation_horizon`/`granularity` reads it from here, not a locally hardcoded number.\n\n`functions/` has no `__init__.py` -- it\'s used as a PEP 420 implicit\nnamespace package (`from functions.Features_Functions import ...` works\ndirectly from the repo root, confirmed against how every notebook actually\nimports it).\n\n## Two model tracks -- and why there are two\n\n**AutoML Forecasting** (notebooks 4 and 5) is the "which approach wins"\nexperimentation track -- it\'s the fastest way to get a strong forecasting\nmodel with zero feature-selection or hyperparameter work, and it\'s the\nbenchmark the comparison sweep in notebook 5 measures every alternative\nagainst.\n\n**A self-trained XGBoost model** (`train_xgboost_candidate_op` inside\n`6_Kubeflow_Pipelines.ipynb`) is what the actual retraining *pipeline* uses,\nnot AutoML. This is a deliberate, practical choice, not an oversight: a KFP\npipeline needs full programmatic control over training, evaluation, and the\nchampion/challenger comparison on every run, and AutoML Forecasting is\ndesigned to be driven interactively (or as one long-running job you poll),\nnot as a fast, cheap, fully scriptable step inside a DAG that also needs to\ncompare against a stored prior WAPE and conditionally deploy. Training a\nplain `XGBRegressor` directly inside the component gives that control (and\ncosts nothing beyond the container\'s own compute) at the price of AutoML\'s\nown search over model architectures -- an explicit tradeoff, not a hidden\none.\n\n## Repo layout\n\n```\nfunctions/              shared package -- feature engineering, metrics, params.yaml loader\nconfig/params.yaml       forecast_horizon / validation_horizon / granularity (uploaded by notebook 1)\n0.._8..ipynb             the pipeline, one numbered notebook per stage (see table above)\nsrc/forecasting/        shared package for the CI/CD scaffold -- currently just data_helpers.py\ntests/                   pytest -- unit tests + a pipeline-compiles check\nci/                      scripts the GitHub Actions workflows call\n.github/workflows/       CI (lint+test) and deploy-pipeline (recompile+push on merge)\ndocs/ARCHITECTURE.md     how the pieces fit together and why some choices were made\n```\n\n`forecasting_pipeline.yaml` (the compiled KFP pipeline template) is a build\nartifact, not source -- `ci/compile_pipeline.py` regenerates it from\n`6_Kubeflow_Pipelines.ipynb` on every CI run, so it\'s gitignored rather than\ncommitted; a committed copy would just go stale the next time the notebook\nchanges.\n\n## Running things locally\n\n```\npip install -r requirements.txt        # to run the notebooks themselves\npip install -r requirements-dev.txt    # to run the CI/CD scaffold\'s own tests\nruff check src/ tests/ ci/\nPYTHONPATH=src pytest tests/ -v\n```\n\n`test_pipeline_compiles.py` is worth understanding on its own: it extracts\nnotebook 6\'s actual component + pipeline code and feeds it to the real `kfp`\ncompiler, so a change that breaks the pipeline\'s DAG (the exact class of bug\nthis project hit early on -- a `PipelineArtifactChannel` with no `.uri`)\nfails CI, not just a future notebook run.\n\n## Recently fixed (real bugs, not hypothetical)\n\nWorth listing plainly, since this project treats "what\'s actually correct"\nas more interesting than "what runs without an error":\n\n- **`2_Features.ipynb`\'s future rows were missing almost every required\n  column.** The batch-prediction input (`future_df.csv`) only populated\n  date-derived columns and left everything else -- `abc_class`, `category`,\n  `price`, ... -- null, which is exactly what produced the\n  `Missing struct property: abc_class` batch-prediction failure. Fixed by\n  carrying every `AVAILABLE_AT_FORECAST` column forward from each SKU\'s last\n  known row (this dataset has no real future pricing/promo plan, so "frozen\n  at the last known value" is the honest simplification) while genuinely\n  recomputing the date-derived ones and leaving every\n  `UNAVAILABLE_AT_FORECAST` column null.\n- **The same ~80-column covariate-typing lists were hand-duplicated** across\n  `2_Features.ipynb`, `4_Baseline_Model.ipynb`, and\n  `5_Model_Comparison_ABC_Category.ipynb`, and `forecast_horizon` was\n  hardcoded independently in three separate places. All three now import\n  `AVAILABLE_AT_FORECAST` / `UNAVAILABLE_AT_FORECAST` / `build_column_specs()`\n  / `FORECAST_HORIZON` from `functions/Features_Functions.py`, which itself\n  reads `forecast_horizon` from `config/params.yaml` -- one number, one\n  place, instead of three copies that could (and had started to) drift.\n- **A real train/serve skew in `batch_forecast_to_bq_op`\'s rolling\n  features.** Training (`add_target_lag_features`) shifts the target by\n  `forecast_horizon` *before* computing `sales_qty_rollmean_4/12` and\n  `rollstd_4/12`, so a training-time rolling feature is anchored 8-11 weeks\n  before its own row, not 0-3 weeks before it. The pipeline\'s recursive\n  batch-forecast component was computing these as a naive "last `w` values\n  seen so far" -- anchored at the wrong offset entirely, feeding the model\n  features it was never trained to see. Fixed to use the same\n  shift-by-`forecast_horizon` anchor as training (confirmed by backtesting\n  the fixed formula against pandas\' own training-time calculation across 8\n  forecast steps -- 32/32 checks match exactly, versus 1/32 for the old\n  formula). A smaller companion bug -- `np.std`\'s default (population, ddof=0)\n  vs. pandas\' rolling `.std()` default (sample, ddof=1) -- was fixed the same\n  way. One consequence worth knowing: because every lag and the (now-correct)\n  rolling anchor land on or before the last real actual for every step up to\n  `forecast_horizon`, this pipeline\'s 8-step forecast never actually needs to\n  recurse on its own predictions under the current horizon -- the recursive\n  code path stays in for correctness if a future run\'s `forecast_horizon`\n  argument ever exceeds the one baked into training, not because today\'s\n  8-step forecast exercises it.\n- **`5_Model_Comparison_ABC_Category.ipynb`\'s local AutoARIMA step ran\n  unconditionally** with no way to skip it. Now gated behind\n  `RUN_LOCAL_ARIMA` (default `True`, since it\'s free), with a\n  reload-from-CSV fallback when flipped off, matching the pattern\n  `RUN_AUTOML` already used later in the same notebook.\n\n## What\'s real vs. placeholder\n\nWorth being direct about, since this is meant to be walked through in\ninterviews:\n\n**Real and tested:** the pipeline\'s plumbing end to end -- DAG dependencies,\nartifact passing, the conditional-deploy logic, the notification wiring.\n`train_xgboost_candidate_op` fits a real `XGBRegressor` on the feature data\nand saves a real model artifact (verified by reloading it with a fresh\n`xgb.Booster()`, the same load path the serving container uses).\n`deploy_model_op` registers and deploys that model to a live Vertex AI\nEndpoint, with retry logic around a transient backend error class hit on a\nreal run. `evaluate_and_compare_op` compares the candidate against the\n*actual* current champion\'s stored validation WAPE (a Model Registry label\nset at deploy time), not a hardcoded number. `batch_forecast_to_bq_op`\ndownloads the *actual* current champion\'s `model.bst` from its Model\nRegistry artifact URI, loads it with a real `xgb.Booster`, and produces a\ngenuine 8-step-ahead forecast per SKU with every lag/rolling feature\ncorrectly anchored to match training (see "Recently fixed" above). It also\nwrites a direct-look CSV -- `gs://<bucket>/data/output/actual_vs_predicted.csv`\n-- with every historical row\'s real actual value next to the champion\'s own\nin-sample prediction for that row, followed by the forecast horizon rows, so\nthe model\'s fit and its forecast are visible in one file without writing SQL\nagainst BigQuery first.\n\n**Still placeholder:** `prepare_data_op`\'s data-refresh step\n(`# df = fetch_fresh_data()`) -- training and batch-forecasting both still\nread a static CSV from GCS, so nothing here reflects genuinely new data. The\none remaining honest simplification in both `2_Features.ipynb`\'s future rows\nand `batch_forecast_to_bq_op`: covariates with no real future plan in this\nsynthetic dataset (price, discount_pct, marketing_spend, ...) are carried\nforward from each SKU\'s last known row rather than a real plan -- the thing\nto revisit once `prepare_data_op` does real refreshes. A real `actuals` feed\nfor the accuracy job, and Terraform for the infra pieces, are the other\nconcrete next steps -- not "someday," genuinely the next things to build.\n\nSee `docs/ARCHITECTURE.md` for how the pieces fit together and why a few of\nthe less-obvious choices were made.\n')
print("wrote README.md")

In [ ]:
pathlib.Path('docs/ARCHITECTURE.md').parent.mkdir(parents=True, exist_ok=True)
pathlib.Path('docs/ARCHITECTURE.md').write_text('# Architecture\n\n## The project, end to end\n\n```mermaid\nflowchart TD\n    N0[0_Setup] --> N1[1_Generate_Synthetic_Data]\n    N1 -->|writes config/params.yaml| P[(params.yaml)]\n    N1 --> N2[2_Features]\n    P -.read by.-> N2\n    N2 --> N3[3_EDA]\n    N2 --> N4[4_Baseline_Model: AutoML]\n    N2 --> N5[5_Model_Comparison: AutoML vs local AutoARIMA vs darts]\n    N5 --> N51[5.1_Experiment_Tracking]\n    N2 --> N6\n\n    subgraph N6[6_Kubeflow_Pipelines -- the retraining pipeline]\n        direction TD\n        A[prepare_data_op] --> B[train_xgboost_candidate_op]\n        B --> C[evaluate_and_compare_op]\n        C -->|candidate wins| D[deploy_model_op]\n        D --> E1[notify_promotion_op -> Pub/Sub]\n        D --> F1[batch_forecast_to_bq_op]\n        C -->|candidate loses| F2[batch_forecast_to_bq_op]\n        F1 --> G[(latest_forecasts / forecast_history)]\n        F2 --> G\n    end\n\n    G --> N7[7_Monitoring_and_Consumption]\n    N7 --> H[Power BI]\n    N7 --> I[forecast_accuracy scheduled query]\n    J[ModelDeploymentMonitoringJob] -.watches live endpoint traffic.-> D\n    N8[8_CICD_and_Packaging] -.scaffolds src/tests/ci/.github, not part of the DAG.-> N6\n```\n\nEverything inside the `6_Kubeflow_Pipelines` box is one KFP pipeline,\nwrapped in a `dsl.ExitHandler` that emails on completion either way. `J`\n(skew/drift monitoring) sits outside the pipeline entirely -- it watches\nthe *endpoint*, not a pipeline run. Notebooks 0-3 and 5/5.1 are upstream of\nthe pipeline (data generation, feature engineering, and the offline\nalgorithm comparison that justified training XGBoost inside the pipeline\nrather than something else); notebook 7 is downstream of it.\n\n## `config/params.yaml` -- one source of truth for the horizon\n\n`1_Generate_Synthetic_Data.ipynb` uploads `gs://<bucket>/config/params.yaml`\nwith three values: `forecast_horizon` (8 weeks), `validation_horizon` (104\nweeks), and `granularity` ("W"). `functions/params_utils.load_params()` is\nthe only function that reads it, and `functions/Features_Functions.py`\nreads it once at import time into `FORECAST_HORIZON` /\n`VALIDATION_HORIZON` / `GRANULARITY` (and derives `LAGS` /\n`ROLLING_WINDOWS` from `GRANULARITY`). Every notebook that needs the\nhorizon -- `2_Features.ipynb`\'s future-row construction,\n`4_Baseline_Model.ipynb` and `5_Model_Comparison_ABC_Category.ipynb`\'s\ntraining-job calls, `6_Kubeflow_Pipelines.ipynb`\'s batch-forecast component\n-- either imports `FORECAST_HORIZON` from `Features_Functions` or receives\nit as a KFP component parameter defaulted to the same value. This replaced\nthree independently hardcoded `forecast_horizon = 8` literals that had\nalready started to drift from each other before being consolidated -- see\nthe README\'s "Recently fixed" section.\n\n## Feature engineering: what\'s available at forecast time, and why it matters\n\nVertex AI Forecasting (and this project\'s own batch-forecast logic) needs\nto know, for every column, whether it can have a real value on a *future*\nrow or not:\n\n- **`AVAILABLE_AT_FORECAST`** (`functions/Features_Functions.py`) -- static\n  product attributes (category, color, brand, ...), calendar features, and\n  covariates this synthetic dataset has no real forward-looking plan for\n  (price, discount_pct, marketing_spend, ...). That last group is carried\n  forward from each SKU\'s last known value on future rows rather than a\n  real plan -- an explicit, honest simplification, not a hidden one; see\n  "Known gaps" below.\n- **`UNAVAILABLE_AT_FORECAST`** -- the target itself, plus every lag/rolling\n  feature and endogenous business signal (reviews, weather, inventory, ...)\n  that genuinely isn\'t known in advance. These stay `null` on future rows;\n  that null is what tells Vertex AI "forecast this row."\n\nBefore this was centralized, `2_Features.ipynb`\'s future-row cell only\npopulated date-derived columns and left the rest -- including `abc_class`\n-- null, which produced a real `Missing struct property: abc_class`\nbatch-prediction failure. It\'s fixed now (see README), but the underlying\nlesson is why these two lists live in one file (`Features_Functions.py`)\nthat every consumer imports, instead of three separate hand-typed copies\nthat can silently stop matching each other.\n\n## Two model tracks, and why `6_Kubeflow_Pipelines.ipynb` doesn\'t use AutoML\n\n`4_Baseline_Model.ipynb` and `5_Model_Comparison_ABC_Category.ipynb` are\nthe experimentation track: they exist to answer "which forecasting\napproach is actually best for this data," sweeping Vertex AI AutoML\nForecasting against a local `statsforecast` AutoARIMA and darts\nN-BEATS/N-HiTS by WAPE, broken out by category and ABC class.\n\nThe retraining *pipeline* (`6_Kubeflow_Pipelines.ipynb`) instead trains a\nplain `XGBRegressor` inside `train_xgboost_candidate_op`. This is a\ndeliberate tradeoff, not an inconsistency: a KFP pipeline needs full\nprogrammatic control on every run -- train, compare against the *actual*\ncurrent champion\'s stored WAPE, and conditionally deploy, all inside one\nscripted DAG. AutoML Forecasting is built to be driven interactively (or\npolled as one long job), not as a fast, cheap, fully scriptable step that\nalso needs to read back a prior run\'s metric and branch on it. Training\nXGBoost directly inside the component gives that control, at the cost of\nAutoML\'s own architecture search -- worth stating plainly rather than\nleaving the two tracks looking like an oversight.\n\n## Decisions worth explaining, not just stating\n\n**Why `batch_forecast_to_bq_op` is called from both branches of the\nconditional, instead of once after it.** KFP\'s control-flow groups\n(`dsl.If`/`dsl.Else`) don\'t allow a task outside the group to depend on\none defined inside it -- the inside task might not have run at all. "Run\nthis exactly once, whichever way the condition went" has to mean calling\nit from both branches, each correctly ordered relative to whatever else\nthat branch does. Confirmed by compiling a minimal throwaway pipeline\nwith this exact shape and inspecting the resulting DAG before touching\nthe real pipeline.\n\n**Why the champion lookup is `Model.list(..., order_by="create_time\ndesc")[0]`, not just `Model.list(...)[0]`.** `Model.list()`\'s default\nordering is unspecified -- confirmed via the SDK\'s own docstring. Every\nretraining run registers a new model under the same `display_name`\n(no `parent_model=` is passed, so each is its own resource, not a new\nversion of one), so without an explicit order, "the champion" would have\nbeen an arbitrary past model, not reliably the current one. A real bug,\ncaught while reusing this exact lookup for the batch-forecast component.\n\n**Why local `statsforecast`/darts replace BQML/AutoML by default.**\nStraightforward cost tradeoff, not a technical constraint -- BQML\ntraining and Vertex AI AutoML Forecasting both cost real money per run\n(AutoML: ~$21/node-hour; BQML: per-TB-scanned x candidate models x\nbacktest windows), and this project\'s GCP credits are limited. AutoML\nstays available behind `RUN_AUTOML = False`, and local AutoARIMA behind\n`RUN_LOCAL_ARIMA = True` (it\'s free, so it defaults on), for when it\'s\nworth spending on that comparison point again.\n\n**Why notifications split into email vs. Pub/Sub instead of one\nchannel.** `dsl.ExitHandler` + `VertexNotificationEmailOp` only knows\n"the pipeline finished" (success or failure) -- it can\'t distinguish "the\ncandidate lost, nothing changed" from "the candidate won and is now\nlive." Those are genuinely different events for different audiences (one\nis an incident signal, one is a deployment signal), so they\'re two\nseparate mechanisms rather than overloading one.\n\n**Why `aiplatform.PipelineJobSchedule` instead of Cloud Scheduler + a\nCloud Function.** Vertex has a native scheduling resource for pipeline\njobs -- one thing to create and maintain, billed the same as a manually\nsubmitted run (nothing extra for the schedule itself), instead of a\nScheduler job whose only purpose is to fire an HTTP endpoint that a\nseparate Cloud Function turns into a pipeline submission.\n\n**Why the realized-vs-predicted accuracy job is a BigQuery Scheduled\nQuery, not another KFP pipeline.** It\'s one SQL join and aggregation over\ntwo tables. Spinning up a full pipeline container to run one query would\nbe paying container-startup overhead for something BigQuery already\nschedules natively, for free. Same reasoning as the point above: match\nthe tool to the actual job, don\'t reach for the heaviest one by default.\n\n**Why CI/CD authenticates with Workload Identity Federation, not a\nservice account key.** A downloaded JSON key is a long-lived credential\nthat has to be stored as a GitHub secret and rotated manually forever;\nWIF lets GitHub Actions impersonate a service account per-run via a\nshort-lived token, with no standing credential to leak or rotate. This is\nGoogle\'s own current recommendation for CI/CD auth, not a preference.\n\n**Why the champion\'s WAPE is stored as a Model Registry label, not read\nfrom MLflow or a metrics file.** `evaluate_and_compare_op` needs the\n*previous* winning run\'s WAPE at comparison time, and the Model resource\nit already looks up (to find the current champion) is the simplest place\nto keep it -- no second system to query, no extra artifact to pass\naround. Labels can\'t hold a `.` (lowercase letters/digits/underscores/\ndashes only, per the SDK\'s own docstring), so the value is stored as basis\npoints (`round(wape * 10000)`) and divided back down on read. A real bug\nthis replaced: the comparison used to be against a hardcoded `0.100`,\nwhich a real (non-synthetic) candidate\'s WAPE will often lose to\nregardless of whether it actually improved on the champion.\n\n**Why `batch_forecast_to_bq_op` builds its own future rows instead of\nreading them from the feature file, and why its lag/rolling features are\nanchored the way they are.** The static feature file has no genuine\nfuture dates -- there\'s no row to predict *against* beyond what already\nexists, so the component constructs `forecast_horizon` (default 8) rows\nper SKU itself. Date-derived features (`week_of_year`, `month`, sin/cos\npairs) are recomputed for the real future date, not carried forward. Lag\nfeatures (`sales_qty_lag_8` and up, plus the four `*_lag8` business-signal\ncolumns) and the rolling features (`rollmean_4/12`, `rollstd_4/12`) both\nhave to land on the *same anchor training used*: `add_target_lag_features`\nshifts the target by `forecast_horizon` before computing lags directly\n(`shift(lag)`, `lag >= forecast_horizon` always) and before rolling\n(`shift(forecast_horizon).rolling(window)`), so a training-time rolling\nfeature at row `p` reflects the window ending at `p - forecast_horizon`,\nnot `p - 1`. The inference-time formula for both is\n`len(history_so_far) - forecast_horizon` as the anchor point -- worked out\nby hand and confirmed by backtesting the fixed formula against pandas\'\nown training-time calculation (32/32 checks matched exactly; the earlier\nnaive "last `w` values seen so far" version matched essentially none of\nthem). Each step\'s own prediction is still appended to a running history\nafterwards: under this project\'s actual `forecast_horizon` (8, matching\nwhat\'s baked into training), every step\'s anchor resolves within real\nhistory and the appended predictions are never actually read back for the\ntarget column -- but the append keeps the code correct in general, for a\nhypothetical future run where the component\'s `forecast_horizon` argument\nexceeds the one training used. The one covariate group still carried\nforward flatly (price, discount_pct, marketing_spend, ...) -- in both this\ncomponent and `2_Features.ipynb`\'s future rows -- is the thing to revisit\nonce `prepare_data_op` does real refreshes and a genuine forward plan for\nthose exists.\n\n## Known gaps -- what\'s still ahead, stated plainly\n\n- **`prepare_data_op` doesn\'t refresh data.** Still the original\n  placeholder (`# df = fetch_fresh_data()`). Training and batch-forecasting\n  both read a static CSV from GCS regardless of what this component does.\n  This is now the single biggest gap -- training, evaluation, deployment,\n  and batch inference are all real; only the data feeding all of them is\n  static.\n- **No `actuals` feed.** The realized-vs-predicted accuracy query reads\n  from a table nothing populates. Needs a real source for demand actuals\n  once they\'re known.\n- **No Terraform / IaC.** Every resource created in these notebooks\n  (Pub/Sub topic, BQ dataset, the pipeline schedule, the monitoring job)\n  was created by hand-run notebook cells, not declared as code. That\'s\n  fine for a portfolio project\'s current stage; it\'s the natural next\n  phase before this could be called a real production setup.\n- **`src/forecasting/` (the CI/CD scaffold\'s own package) has exactly one\n  function in it.** `data_helpers.py` is the first piece of what the\n  roadmap calls a shared package for that scaffold specifically -- separate\n  from `functions/`, which is the real, already-multi-file package the\n  notebooks themselves import. Notebook 5\'s own `fill_series_gaps` still\n  isn\'t wired to import `src/forecasting/data_helpers.py`\'s copy;\n  migrating it is a deliberate follow-up, not done as part of this pass.\n- **`5.1_Experiment_Tracking.ipynb`\'s `get_champion_metrics` is still a\n  draft, not a wired-in KFP component.** It\'s commented out\n  (`# @dsl.component(...)`) and references `PROJECT_ID`/`REGION` as if\n  they were notebook globals, which won\'t resolve inside an actual\n  isolated component container. Turning it into a real\n  `@dsl.component` with those passed as parameters (matching every other\n  component in `6_Kubeflow_Pipelines.ipynb`) is the next step before it\n  could be added to the pipeline\'s DAG.\n')
print("wrote docs/ARCHITECTURE.md")

### Confirm the lint step passes too

Same command the CI workflow runs.

In [ ]:
import subprocess

result = subprocess.run(["ruff", "check", "src/", "tests/", "ci/"], capture_output=True, text=True)
print(result.stdout or result.stderr)